[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/conformal_elliptical/conformal_elliptical.ipynb)

# Planes Across the Cube's Rotation Axes

This example draws the thirteen great circles cut on the unit sphere by the planes perpendicular to a cube's rotation axes and verifies that the cube's rotation group permutes them. Three of those axes pass through opposite face centres, six pass through opposite edge midpoints, and four pass through opposite corners. Nine of the thirteen central planes, perpendicular to the face and edge axes, also act as mirror planes of the cube, while every proper rotation of the cube carries the full set of thirteen great circles onto itself.

In 3D elliptic geometric algebra, a central plane is modelled as an extrinsic vector and a point on the unit sphere is modelled as an intrinsic antivector, so their pairing `planes & points` gives each point's signed height above each plane and vanishes along the great circle. Rotating the planes with the sandwich product `generators >> planes` and taking the metric inner product `turned_planes | planes` yields unit magnitude whenever a rotated plane coincides with one of the thirteen planes up to orientation.

In Gibbs–Heaviside vector and matrix notation, each central plane is represented by a unit normal $\hat{\mathbf{n}}_i \in \mathbb{R}^3$, its signed height at a hemisphere point $\mathbf{x} = (x, y, \sqrt{1 - x^2 - y^2})^\top$ is $\hat{\mathbf{n}}_i^\top \mathbf{x}$, and planes transform as covectors under inverse-transpose matrices $(A^{-1})^\top \hat{\mathbf{n}}_i$. Generating the octahedral rotation group $O \subset \mathrm{SO}(3)$ about unit axis $\hat{\mathbf{u}}$ by angle $\theta$ builds the Rodrigues matrix $R(\hat{\mathbf{u}}, \theta) = I\cos\theta + (1 - \cos\theta)\hat{\mathbf{u}}\hat{\mathbf{u}}^\top + [\hat{\mathbf{u}}]_\times\sin\theta$ and checks $|\hat{\mathbf{n}}_j^\top R\hat{\mathbf{n}}_i| = 1$ across pairs of normals.

In [ ]:
# Install numga and its examples when running in Colab.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
import numpy as np
from numga import Algebra, NumpyContext, stack

from examples.quadrics.conformal_elliptical import render
from examples.util.animation import save_figure

np.set_printoptions(precision=4, suppress=True)

ga = Algebra("x+y+z+")
mv = NumpyContext(ga).multivector
Plane = ga.gatype.vector()                          # Vector+

## Central planes across the rotation axes

Each rotation axis of the cube defines a central plane perpendicular to that axis, cutting the unit sphere in a great circle.

In [ ]:
# Integer direction vectors along the three face, six edge, and four corner rotation axes of the cube.
face_axes = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]
edge_axes = [[1, 1, 0], [1, -1, 0], [1, 0, 1], [1, 0, -1], [0, 1, 1], [0, 1, -1]]
corner_axes = [[1, 1, 1], [1, 1, -1], [1, -1, 1], [-1, 1, 1]]

planes = mv.vector(face_axes + edge_axes + corner_axes).normalized()            # [13] Plane+
save_figure(render.planes_figure(planes), "cube_axis_planes");

Alternating white and grey regions on the front hemisphere track the sign parity of `planes & points` across all thirteen central planes, changing shade across every great circle. Four great circles meet at the central face axis to form an eight-sector star, while six-circle junctions mark the edge axes along the horizontal and vertical meridians.

## Closure under the cube's rotations

A quarter turn in a face plane and a third turn around a body diagonal generate the proper rotation group of the cube. Because both generators preserve the cube, each carries every central plane onto another plane of the thirteen up to orientation, permuting the face, edge, and corner families within themselves.

In [ ]:
quarter_turn = (mv.xy * (np.pi / 4)).exp()                                      # [] Rotor+
third_turn = ((mv.xy + mv.yz + mv.zx).normalized() * (np.pi / 3)).exp()         # [] Rotor+
generators = stack([quarter_turn, third_turn])                                  # [2] Rotor+
turned_planes = generators[:, None] >> planes                                   # [2, 13] Plane+
alignments = (turned_planes[:, :, None] | planes).abs()                         # [2, 13, 13] Scalar+
print("plane permutations under quarter turn and third turn:\n", alignments.argmax(axis=-1))

In [ ]:
# checks
# All thirteen planes have unit norm and remain pairwise distinct up to orientation.
assert ((planes.scalar_norm_squared() - 1).abs() < 1e-6).all()
assert ((planes[:, None] | planes).abs() > 1 - 1e-6).sum() == 13
# Each generating rotation carries every plane onto one of the thirteen planes up to orientation.
assert ((alignments.max(axis=-1) - 1).abs() < 1e-6).all()